# Silver

## Import Helper Functions

In [1]:
from src.config_loader import load_config
from src.spark_sql_magic import sql

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
BRONZE_NAMESPACE = cfg["general"]["namespaces"]["bronze"]
SILVER_NAMESPACE = cfg["general"]["namespaces"]["silver"]
MONITORING_NAMESPACE = cfg["general"]["namespaces"]["monitoring"]
TOPIC_SUFFIX = cfg["kafka"]["topic_suffix"]
LOOKUP_TABLES = set(cfg["raw"]["lookup_tables"].keys())

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["silver"])
        .getOrCreate()
)

## Read from Bronze (Incremental)

In [4]:
def read_bronze_incremental(tablename: str) -> DataFrame:

    bronze_table = f"{CATALOG}.{BRONZE_NAMESPACE}.{tablename}"
    silver_table = f"{CATALOG}.{SILVER_NAMESPACE}.{tablename}"

    if not spark.catalog.tableExists(silver_table):
        return spark.table(bronze_table)

    return spark.sql(f"""
        SELECT *
        FROM {bronze_table}
        WHERE spark_ingest_ts >
        COALESCE(
            (SELECT MAX(spark_ingest_ts) FROM {silver_table}),
            TIMESTAMP '1970-01-01'
        )
    """)

## Common Function 1: Normalize CDC

In [5]:
def normalize_cdc(df: DataFrame) -> DataFrame:
    """
    1 normalize CDC operation
    2 rename CDC timestamp
    3 convert CDC timestamp to real timestamp
    """
    return (
        df
        .withColumn("cdc_op",
            F.when(F.col("__op") == "r", F.lit("c"))
             .otherwise(F.col("__op"))
        )       
        .withColumnRenamed("__ts_ms", "cdc_ts_ms")
        .withColumn("cdc_ts", F.to_timestamp(F.col("cdc_ts_ms") / 1000))
        .withColumn("ds", F.to_date("cdc_ts"))   # easier to query downstream
    )

## Common Function 2: Normalize Decimal Struct

In [6]:
from pyspark.sql.types import StructType, BinaryType, IntegerType, DecimalType

def normalize_decimal_structs(df: DataFrame) -> DataFrame:
    """
    Convert Avro-decoded decimal structs to numeric types.
    Looks for columns with schema: StructType(scale:int, value:binary)

    df (DataFrame)
     └ df.schema  → StructType
           └ fields  → list of StructField
                 └ field.name       → column name
                 └ field.dataType  → column type (IntegerType, StringType, StructType, etc)
                       └ if StructType: field.dataType.fields → inner list of StructField    
    """
    # Step 1: collect all decimal struct columns
    decimal_cols = []

    for field in df.schema.fields:
        if not isinstance(field.dataType, StructType):
            continue  # skip non-struct columns

        # get inner field names and types
        inner_fields = {f.name: f.dataType for f in field.dataType.fields}

        # detect decimal struct pattern
        if inner_fields.get("scale") == IntegerType() and inner_fields.get("value") == BinaryType():
            decimal_cols.append(field.name)

    # Step 2: apply normalization for each detected column
    for col_name in decimal_cols:
        df = df.withColumn(
                col_name,
                (F.conv(F.hex(F.col(f"{col_name}.value")), 16, 10).cast(DecimalType(38,0)) /
                F.pow(10, F.col(f"{col_name}.scale")))
                .cast(DecimalType(18,6))
            )

    return df

## Common Function 3: Drop Kafka Metadata

In [7]:
def drop_kafka_metadata(df: DataFrame) -> DataFrame:
    kafka_metadata_cols = [col for col in df.columns if col.startswith("kafka_")]
    if kafka_metadata_cols:
        return df.drop(*kafka_metadata_cols)
    return df

## Function: Apply Common Transform

In [8]:
def apply_common_transform(df: DataFrame) -> DataFrame:
    df = normalize_cdc(df)
    df = normalize_decimal_structs(df) 
    df = drop_kafka_metadata(df)
    return df

## Function: Apply Table Specific Transform

In [9]:
from src.silver.transform import TABLE_TRANSFORMS

def apply_table_specific_transform(df: DataFrame, tablename: str) -> DataFrame:
    transform_fn = TABLE_TRANSFORMS.get(tablename)
    if transform_fn is None:
        raise ValueError(
            f"No table-specific transform registered for '{tablename}'. "
            f"Available: {sorted(TABLE_TRANSFORMS)}"
        )
    return transform_fn(df)

## Function: Apply Schema Drift

In [10]:
from pyspark.sql.utils import AnalysisException

def apply_schema_drift(silver_ready_df, silver_table_name):
    """
    Detects table existence, handles additive schema evolution,
    and prevents destructive schema changes.
    """
    # ---- Detect table existence ----
    try:
        silver_existing_df = spark.table(silver_table_name)
    except AnalysisException:
        return False   # table does not exist
    
    # ---- Drift Detection ----
    incoming_cols = set(silver_ready_df.columns)
    existing_cols = set(silver_existing_df.columns)
    
    new_cols = incoming_cols - existing_cols
    missing_cols = existing_cols - incoming_cols
    
    if missing_cols:
        raise Exception(f"Destructive schema change detected: {missing_cols}")
    
    # ---- Additive evolution ----
    for col in new_cols:
        dtype = silver_ready_df.schema[col].dataType.simpleString()
        spark.sql(f"""
            ALTER TABLE {silver_table_name} 
            ADD COLUMN {col} {dtype}
        """)

    return True

## DDL: Create Table DQ Metrics

In [11]:
%%sql
CREATE TABLE IF NOT EXISTS monitoring.dq_metrics (
    pipeline_stage STRING,
    source_table STRING,
    metric_name STRING,
    metric_value BIGINT,
    timestamp TIMESTAMP
)
USING ICEBERG
PARTITIONED BY (pipeline_stage, source_table)

## Function: Collect Table DQ Metrics

In [12]:
from src.silver.dq import TABLE_DQ_METRICS
from src.silver.dq.common_dq import collect_common_dq_metrics

def collect_table_dq_metrics(df, table_name: str):
    """
    For CDC tables:
      - run common DQ + table-specific DQ
    For lookup snapshot tables:
      - run table-specific DQ only (skip common CDC checks)
    """
    table_dq_fn = TABLE_DQ_METRICS.get(table_name)
    if table_dq_fn is None:
        raise ValueError(
            f"No table-specific DQ metrics function registered for '{table_name}'. "
            f"Available: {sorted(TABLE_DQ_METRICS)}"
        )
    if table_name in LOOKUP_TABLES:
        # lookup tables don't have cdc_op/cdc_ts/spark_ingest_ts/batch_id
        table_dq_fn(df, table_name)
    else:
        collect_common_dq_metrics(df, table_name)
        table_dq_fn(df, table_name)

    return df

## DDL: Create Table Pipeline Audit

In [13]:
%%sql
CREATE TABLE IF NOT EXISTS monitoring.pipeline_audit (
    pipeline_stage STRING,
    source_table STRING,
    target_table STRING,
    run_ts TIMESTAMP,
    input_rows BIGINT,
    output_rows BIGINT,
    status STRING,
    error_message STRING
)
USING ICEBERG
PARTITIONED BY (pipeline_stage, source_table)

## Function: Log Pipeline Audit

In [14]:
from datetime import datetime

def log_pipeline_audit(
    source_table: str,
    target_table: str,
    input_rows: int,
    output_rows: int,
    status: str,
    error_message: str | None = None
):

    error_message = error_message or ""

    audit_df = spark.createDataFrame(
        [
            (
                "silver",                 # pipeline stage
                source_table,             # bronze source
                target_table,             # silver target
                datetime.utcnow(),        # run timestamp
                int(input_rows),
                int(output_rows),
                status,
                error_message,
            )
        ],
        [
            "pipeline_stage",
            "source_table",
            "target_table",
            "run_ts",
            "input_rows",
            "output_rows",
            "status",
            "error_message",
        ],
    )

    audit_df.writeTo(
        f"{CATALOG}.{MONITORING_NAMESPACE}.pipeline_audit"
    ).append()

## Append only model (Functional DE)
I know many would question this code below so I included the reference.
### Reference:
- [The Data Warehouse Setup No One Taught You](https://blog.dataexpert.io/p/the-data-warehouse-setup-no-one-taught)
- [Functional Data Engineering — a modern paradigm for batch data processing](https://maximebeauchemin.medium.com/functional-data-engineering-a-modern-paradigm-for-batch-data-processing-2327ec32c42a)
- [SCD 2 Implemetation Strategies](https://github.com/SoongGuanLeong/data_pipelines_batch_stream_vector/blob/main/1-batch/docs/architecture/gold/scd2_implementation_strategies.md)

In [15]:
import traceback

def run_silver_pipeline(table_name):

    silver_table_name = f"{CATALOG}.{SILVER_NAMESPACE}.{table_name}"
    input_rows = 0
    output_rows = 0

    try:
        # -------------------------
        # A) LOOKUP SNAPSHOT FLOW
        # -------------------------
        if table_name in LOOKUP_TABLES:
            # lookup tables are snapshot-style, not CDC-style
            bronze_table_name = f"{CATALOG}.{BRONZE_NAMESPACE}.{table_name}"
            df = spark.table(bronze_table_name)
    
            input_rows = df.count()
            df = apply_table_specific_transform(df, table_name)
            _ = apply_schema_drift(df, silver_table_name)
            df = collect_table_dq_metrics(df, table_name)
    
            # usually overwrite for lookup snapshots
            df.repartition("ds").write.format("iceberg").mode("overwrite").saveAsTable(silver_table_name)
    
            output_rows = spark.table(silver_table_name).count()
            log_pipeline_audit(table_name, silver_table_name, input_rows, output_rows, "success")
            return

        # -------------------------
        # B) CDC INCREMENTAL FLOW
        # -------------------------
        df = read_bronze_incremental(table_name)
        
        input_rows = df.count()
        
        df = apply_common_transform(df)
        df = apply_table_specific_transform(df, table_name)
        table_exists = apply_schema_drift(df, silver_table_name)
        df = collect_table_dq_metrics(df, table_name)
        df = df.repartition("ds")                                               # when table becomes bigger, need to repartition more columns
        
        output_rows = input_rows
        
        writer = (
            df.writeTo(silver_table_name)                                       # DataFrameWriterV2
            .using("iceberg")
            .tableProperty("format-version", "3")
            .tableProperty("write.format.default", "parquet")
            .tableProperty("write.parquet.compression-codec", "zstd")
            .tableProperty("write.target-file-size-bytes", "134217728")         # <- this depends on daily data size
            .partitionedBy("ds")                                                # can also F.partitioning.days("cdc_ts")
        )
        
        if table_exists:
            writer.append()
        else:
            writer.create() 

        log_pipeline_audit(
            table_name,
            silver_table_name,
            input_rows,
            output_rows,
            "SUCCESS"
        )

    except Exception as e:
        error_message = traceback.format_exc()[:2000]
        
        log_pipeline_audit(
            table_name,
            silver_table_name,
            input_rows,
            0,
            "FAILED",
            error_message
        )

        raise

In [16]:
all_tables = list(TABLE_TRANSFORMS.keys())
cdc_tables = [t for t in all_tables if t not in LOOKUP_TABLES]
lookup_tables = [t for t in all_tables if t in LOOKUP_TABLES]

for table_name in cdc_tables + lookup_tables:
    print(f"Running silver transform for: {table_name}")
    run_silver_pipeline(table_name)

Running silver transform for: customers


Running silver transform for: order_items


Running silver transform for: order_payments


Running silver transform for: order_reviews


Running silver transform for: orders


Running silver transform for: products


Running silver transform for: sellers
Running silver transform for: geolocation


Running silver transform for: product_category_name


## Sanity Check

In [17]:
w = W.partitionBy("pipeline_stage", "source_table", "metric_name").orderBy(F.col("timestamp").desc())

dq_df = (
        spark.table(f"{CATALOG}.{MONITORING_NAMESPACE}.dq_metrics")
        .withColumn("row_num", F.row_number().over(w))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

for table_name in TABLE_TRANSFORMS:
    df = dq_df.filter(F.col("source_table") == table_name)
    df.show(n=50)

+--------------+------------+--------------------+------------+--------------------+
|pipeline_stage|source_table|         metric_name|metric_value|           timestamp|
+--------------+------------+--------------------+------------+--------------------+
|        silver|   customers|          bad_cdc_op|           0|2026-04-03 05:21:...|
|        silver|   customers|     duplicate_count|           0|2026-04-03 05:21:...|
|        silver|   customers|       future_cdc_ts|           0|2026-04-03 05:21:...|
|        silver|   customers|   future_created_at|           0|2026-04-03 05:21:...|
|        silver|   customers|   future_updated_at|           0|2026-04-03 05:21:...|
|        silver|   customers| invalid_customer_id|           0|2026-04-03 05:21:...|
|        silver|   customers|invalid_customer_...|           0|2026-04-03 05:21:...|
|        silver|   customers|invalid_customer_...|           0|2026-04-03 05:21:...|
|        silver|   customers|       null_batch_id|           0|20

In [18]:
%%sql
SELECT COUNT(*) FROM polaris.silver.customers

+--------+
|count(1)|
+--------+
|99441   |
+--------+



In [19]:
%%sql
SELECT * FROM polaris.monitoring.pipeline_audit

+--------------+---------------------+------------------------------------+--------------------------+----------+-----------+-------+-------------+
|pipeline_stage|source_table         |target_table                        |run_ts                    |input_rows|output_rows|status |error_message|
+--------------+---------------------+------------------------------------+--------------------------+----------+-----------+-------+-------------+
|silver        |sellers              |polaris.silver.sellers              |2026-04-03 05:22:04.400837|3095      |3095       |SUCCESS|             |
|silver        |geolocation          |polaris.silver.geolocation          |2026-04-03 05:22:12.465042|1000163   |19015      |success|             |
|silver        |order_reviews        |polaris.silver.order_reviews        |2026-04-03 05:21:48.501207|98410     |98410      |SUCCESS|             |
|silver        |product_category_name|polaris.silver.product_category_name|2026-04-03 05:22:14.588501|71        

In [20]:
for table_name in TABLE_TRANSFORMS:
    df = spark.table(f"{CATALOG}.{SILVER_NAMESPACE}.{table_name}")
    df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- _op: string (nullable = true)
 |-- cdc_ts_ms: long (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- spark_ingest_ts: timestamp (nullable = true)
 |-- cdc_op: string (nullable = true)
 |-- cdc_ts: timestamp (nullable = true)
 |-- ds: date (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(18,6) (nullable = true)
 |-- freight_value: decimal(18,6) (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: tim

In [21]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop()  